In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [2]:
%load_ext cudf.pandas

/opt/conda/envs/rapids-25.02/lib/python3.10/site-packages/cudf/pandas/__init__.py:65: UserWarning: cudf.pandas detected an already configured memory resource, ignoring 'CUDF_PANDAS_RMM_MODE'=managed_pool
  warnings.warn(


In [3]:
%LoadCheckpoint /home/dias-benchmarks/tpch/notebooks/q06/annotated/checkpoints/pre_cell_1.pickle

trying: ['DATA_ROOT']
me:  0
trying: ['lineitem']


me:  1
trying: ['load_lineitem']
me:  1
trying: ['tpch_parent']
me:  0
trying: ['pd']
me:  0
trying: ['STORAGE_OPTS']
me:  0


Declaring variable DATA_ROOT
Declaring variable tpch_parent
Declaring variable pd
Declaring variable STORAGE_OPTS
Declaring variable lineitem
Declaring variable load_lineitem


In [4]:
%%RecordEvent
%%time
### cell 1 ###

# Filter and compute total entirely on GPU
date_filter = (lineitem["L_SHIPDATE"].dt.year == 1996)

discount_filter = lineitem["L_DISCOUNT"].between(0.08, 0.1)
quantity_filter = lineitem["L_QUANTITY"] < 24

mask = date_filter & discount_filter & quantity_filter

total = (lineitem["L_EXTENDEDPRICE"] * lineitem["L_DISCOUNT"])[mask].sum()

CPU times: user 24.8 ms, sys: 29.4 ms, total: 54.2 ms
Wall time: 68.6 ms


In [5]:
%Checkpoint /home/dias-benchmarks/tpch/notebooks/q06/rewritten/o4_mini_high/checkpoints/post_cell_1_try_0.pickle

migration speed (bps): 800476705.128875
---------------------------
variables to migrate:
discount_filter 48
mask 48
lineitem 48
pd 72
quantity_filter 48
load_lineitem 144
tpch_parent 54
DATA_ROOT 80
total 32
date_filter 48
STORAGE_OPTS 64
---------------------------
variables to recompute:
[]
---------------------------
cells to recompute:
[]
Checkpoint saved to: /home/dias-benchmarks/tpch/notebooks/q06/rewritten/o4_mini_high/checkpoints/post_cell_1_try_0.pickle


In [6]:
%PrintCellInfo opt_cell_exec_info

======= Cell 0 =======
Input variables []
Active variables ['lineitem']
Intermediate variables []
Future variables []
Modified dataframes
======= Cell 1 =======
Input variables []
Active variables []
Intermediate variables ['date_filter', 'total', 'quantity_filter', 'discount_filter', 'mask']
Future variables []
Modified dataframes
Saved cell execution info to opt_cell_exec_info


In [7]:

with open("/home/dias-benchmarks/tpch/notebooks/q06/opt_cell_exec_info_1_try_0.pkl", "wb") as f:
    pickle.dump(opt_cell_exec_info[1], f)


In [ ]:
opt_output = Out.get(4)

In [ ]:
total_opt = total
%LoadCheckpoint /home/dias-benchmarks/tpch/notebooks/q06/annotated/checkpoints/post_cell_1.pickle
assert total_opt == total

import numpy as np
import cudf
from elastic.core.common.pandas import is_type_styler
is_orig_output_pd = isinstance(orig_output, (pd.Series, pd.DataFrame, pd.Index))
is_opt_output_pd = isinstance(opt_output, (pd.Series, pd.DataFrame, pd.Index))
is_orig_output_array = isinstance(orig_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray))
is_opt_output_array = isinstance(opt_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray))
is_orig_output_styler = is_type_styler(type(orig_output))
is_opt_output_styler = is_type_styler(type(opt_output))
if is_orig_output_styler and is_opt_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_orig_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_opt_output_styler:
    assert opt_output.to_html() == orig_output

if is_orig_output_pd and is_opt_output_pd:
    assert orig_output.equals(opt_output)
# TODO(jie): this is a hack.
elif ((is_orig_output_pd or is_opt_output_pd) and (is_orig_output_array or is_opt_output_array)) or (is_orig_output_array and is_opt_output_array):
    assert list(orig_output) == list(opt_output)
else:
    assert orig_output == opt_output
